<a href="https://colab.research.google.com/github/anamitra-tech/ML-Projects/blob/main/AntiDroneGNB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Clear all variables
%reset -f

# Clear imports cache
import sys
sys.modules.clear()

In [ ]:
import os
import glob

# Remove all old CSV/RF data
files = glob.glob('/content/**/*.csv', recursive=True) + glob.glob('/content/**/*.rar', recursive=True)
for f in files:
    os.remove(f)

In [ ]:
!rm -rf "/content/drive/MyDrive/DroneRF"
!mkdir "/content/drive/MyDrive/DroneRF"

In [2]:
!unzip -o "/content/drive/MyDrive/f4c2b4n755-1.zip" -d "/content/drive/MyDrive/DroneRF"

Archive:  /content/drive/MyDrive/f4c2b4n755-1.zip
  inflating: /content/drive/MyDrive/DroneRF/DroneRF/Background RF activites/RF Data_00000_L1.rar  
  inflating: /content/drive/MyDrive/DroneRF/DroneRF/Background RF activites/RF Data_00000_H1.rar  
  inflating: /content/drive/MyDrive/DroneRF/DroneRF/Background RF activites/RF Data_00000_L2.rar  
  inflating: /content/drive/MyDrive/DroneRF/DroneRF/Background RF activites/FR Data_00000_H2.rar  
  inflating: /content/drive/MyDrive/DroneRF/DroneRF/Bepop drone/RF Data_10000_L.rar  
  inflating: /content/drive/MyDrive/DroneRF/DroneRF/Bepop drone/RF Data_10000_H.rar  
  inflating: /content/drive/MyDrive/DroneRF/DroneRF/Bepop drone/RF Data_10011_L.rar  
  inflating: /content/drive/MyDrive/DroneRF/DroneRF/Bepop drone/RF Data_10011_H.rar  
  inflating: /content/drive/MyDrive/DroneRF/DroneRF/Bepop drone/RF Data_10010_L.rar  
  inflating: /content/drive/MyDrive/DroneRF/DroneRF/Bepop drone/RF Data_10001_L.rar  
  inflating: /content/drive/MyDrive/Dr

In [3]:
!pip install rarfile

In [4]:
import rarfile
import os

base = "/content/drive/MyDrive/DroneRF"

count = 0

for root, dirs, files in os.walk(base):
    for file in files:
        if file.endswith(".rar"):
            path = os.path.join(root, file)
            print("Extracting:", path)
            try:
                with rarfile.RarFile(path) as rf:
                    rf.extractall(root)
                count += 1
            except Exception as e:
                print("Error:", e)

print("Total RAR extracted:", count)

Extracting: /content/drive/MyDrive/DroneRF/DroneRF/Background RF activites/RF Data_00000_L1.rar
Extracting: /content/drive/MyDrive/DroneRF/DroneRF/Background RF activites/RF Data_00000_H1.rar
Extracting: /content/drive/MyDrive/DroneRF/DroneRF/Background RF activites/RF Data_00000_L2.rar
Extracting: /content/drive/MyDrive/DroneRF/DroneRF/Background RF activites/FR Data_00000_H2.rar
Extracting: /content/drive/MyDrive/DroneRF/DroneRF/Bepop drone/RF Data_10000_L.rar
Extracting: /content/drive/MyDrive/DroneRF/DroneRF/Bepop drone/RF Data_10000_H.rar
Extracting: /content/drive/MyDrive/DroneRF/DroneRF/Bepop drone/RF Data_10011_L.rar
Extracting: /content/drive/MyDrive/DroneRF/DroneRF/Bepop drone/RF Data_10011_H.rar
Extracting: /content/drive/MyDrive/DroneRF/DroneRF/Bepop drone/RF Data_10010_L.rar
Extracting: /content/drive/MyDrive/DroneRF/DroneRF/Bepop drone/RF Data_10001_L.rar
Error: Failed the read enough data: req=65536 got=21613
Extracting: /content/drive/MyDrive/DroneRF/DroneRF/Bepop drone

In [11]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║   CONTINUAL RF DRONE MONITORING SYSTEM — PRODUCTION GRADE                   ║
# ║   Version 3.0  |  Bayesian Confidence Engine  |  No Retraining              ║
# ║                                                                              ║
# ║   ARCHITECTURE (7 Stages):                                                   ║
# ║                                                                              ║
# ║   Raw RF Signal (CSV file / SDR ring buffer)                                 ║
# ║        │                                                                     ║
# ║   ┌────▼────────────────────────────────────────────────────────────┐        ║
# ║   │ STAGE 1 — Feature Extraction (30 features per window)           │        ║
# ║   │  Window=1024, Step=256                                          │        ║
# ║   │  A: amplitude stats (8)  B: Hilbert envelope (6)               │        ║
# ║   │  C: Welch PSD spectral (8)  D: inst. freq (4)  E: band (4)     │        ║
# ║   └────┬────────────────────────────────────────────────────────────┘        ║
# ║        │ 30-feature vector                                                   ║
# ║   ┌────▼────────────────────────────────────────────────────────────┐        ║
# ║   │ STAGE 2 — Dual Bayesian Classifier (trained ONCE, then FROZEN)  │        ║
# ║   │  Random Forest  → P_RF(class|x)                                 │        ║
# ║   │  GaussianNB     → P_GNB(class|x)  ← true Bayesian posterior    │        ║
# ║   │  Online GNB     → partial_fit() Bayesian update curve           │        ║
# ║   │  Combined posterior = geometric mean of RF × GNB                │        ║
# ║   │  conf ≥ 0.65 → KNOWN DRONE  |  conf < 0.65 → ANOMALY PATH      │        ║
# ║   └────┬──────────────────────────────┬──────────────────────────── ┘        ║
# ║   KNOWN │                        UNKNOWN│                                    ║
# ║   ┌────▼──────────────┐      ┌─────────▼───────────────────────────┐        ║
# ║   │ FRIENDLY_DRONE    │      │ STAGE 3 — 4-Detector Anomaly Gate   │        ║
# ║   │ BACKGROUND        │      │  Mahalanobis  (class-cond. dist)    │        ║
# ║   │ + full Bayesian   │      │  Per-Class GMM (density model)      │        ║
# ║   │   confidence      │      │  Isolation Forest (geometric)       │        ║
# ║   │   breakdown       │      │  GNB Log-Likelihood (Bayesian LL)   │        ║
# ║   └───────────────────┘      └─────────┬───────────────────────────┘        ║
# ║                                        │                                     ║
# ║                              ┌─────────▼───────────────────────────┐        ║
# ║                              │ STAGE 4 — Bayesian Threat Score      │        ║
# ║                              │  Weighted ensemble → score ∈ [0,1]  │        ║
# ║                              │  + Epistemic / Aleatoric split       │        ║
# ║                              │  + Calibrated confidence             │        ║
# ║                              │  + Dirichlet Process GMM novelty     │        ║
# ║                              └──────┬──────────────┬───────────────┘        ║
# ║                         HIGH TS     │         LOW TS│                        ║
# ║                    ┌────────────────┘               │                        ║
# ║                    │                                │                        ║
# ║           ┌────────▼──────────┐     ┌──────────────▼──────────────────┐     ║
# ║           │ POTENTIAL_THREAT  │     │ STAGE 5 — Temporal Trust Engine  │     ║
# ║           │ CONFIRMED_THREAT  │     │  EmitterRecord ring buffer       │     ║
# ║           └───────────────────┘     │  Harmonic-mean trust score       │     ║
# ║                                     │  obs_trust × stab_trust × safe_t │     ║
# ║                                     └──────────────┬───────────────────┘     ║
# ║                                          STABLE    │  UNSTABLE               ║
# ║                                     ┌──────────────▼──────────────────┐     ║
# ║                                     │ STAGE 6 — RF Fingerprint DB      │     ║
# ║                                     │  trusted_db + suspicious_db      │     ║
# ║                                     │  Cosine similarity matching      │     ║
# ║                                     │  JSON-persistent, grows over time│     ║
# ║                                     └──────────────┬───────────────────┘     ║
# ║                                                    │                         ║
# ║                                     ┌──────────────▼──────────────────┐     ║
# ║                                     │ STAGE 7 — Final Decision         │     ║
# ║                                     │  + Full Bayesian confidence      │     ║
# ║                                     │    breakdown per prediction       │     ║
# ║                                     │  FRIENDLY | BACKGROUND |         │     ║
# ║                                     │  POTENTIAL_THREAT |               │     ║
# ║                                     │  CONFIRMED_THREAT |               │     ║
# ║                                     │  SAFE_NEW_DRONE |                 │     ║
# ║                                     │  TRUSTED_NEW_DRONE |              │     ║
# ║                                     │  UNKNOWN_MONITOR                  │     ║
# ║                                     └──────────────────────────────────┘     ║
# ║                                                                              ║
# ║   Dataset: data.mendeley.com/datasets/f4c2b4n755/1                          ║
# ║   Classes: AR drone | Phantom drone | Background RF activities               ║
# ║   Goal:    CONFIDENCE in prediction, not just accuracy                       ║
# ╚══════════════════════════════════════════════════════════════════════════════╝
#
# ─────────────────────────────────────────────────────────────────────────────
# COLAB SETUP  (run these two cells BEFORE this script)
# ─────────────────────────────────────────────────────────────────────────────
#   from google.colab import drive
#   drive.mount('/content/drive')
#
#   !unzip -q "/content/drive/MyDrive/f4c2b4n755-1.zip" \
#             -d "/content/drive/MyDrive/DroneRF"
#   # OR for RAR:
#   # !apt-get install -qq unrar
#   # !unrar x "/content/drive/MyDrive/DroneRF.rar" \
#   #          "/content/drive/MyDrive/DroneRF/"
#   !find "/content/drive/MyDrive/DroneRF" -name "*.csv" | head -10
#
# ─────────────────────────────────────────────────────────────────────────────
# EXPECTED FOLDER LAYOUTS  (BOTH auto-detected)
# ─────────────────────────────────────────────────────────────────────────────
#   Layout A — named subfolders:
#     DroneRF/  AR drone/   Phantom drone/   Background RF activities/
#
#   Layout B — flat folder, BUI-named files:
#     DroneRF/  00000L1.csv  10001H3.csv  11010L2.csv
# ─────────────────────────────────────────────────────────────────────────────


# ════════════════════════════════════════════════════════════════════════════
# SECTION 0 — CONFIGURATION
# ════════════════════════════════════════════════════════════════════════════
DATA_DIR    = "/content/drive/MyDrive/DroneRF/DroneRF"
OUTPUT_CSV  = "dronerf_features_5k.csv"
DB_PATH     = "rf_fingerprint_db.json"
RANDOM_SEED = 42

# ── Feature extraction ────────────────────────────────────────────────────
WINDOW_SIZE      = 200
STEP_SIZE        = 100
TARGET_TOTAL     = 4500  # total segments (~15 min)
CHUNK_BYTES      = 32*1024*1024
MAX_SEGMENTS_PER_FILE = 98

# ── Classifier / anomaly thresholds ──────────────────────────────────────
CONFIDENCE_THRESHOLD   = 0.65   # combined posterior → known vs unknown
HIGH_THREAT_THRESHOLD  = 0.85   # immediate threat flag
TRUST_MIN_OBSERVATIONS = 12     # min sightings before considering safe
TRUST_MAX_VARIANCE     = 0.15   # max feature variance for "stable"
SIMILARITY_THRESHOLD   = 0.85   # cosine sim for fingerprint DB lookup

# ── Anomaly ensemble weights (must sum to 1.0) ────────────────────────────
ANOMALY_WEIGHTS = {
    "mahal":     0.35,
    "gmm":       0.30,
    "isoforest": 0.25,
    "gnb":       0.10,
}


# ════════════════════════════════════════════════════════════════════════════
# SECTION 1 — INSTALL & IMPORT
# ════════════════════════════════════════════════════════════════════════════
import subprocess, sys
subprocess.run(
    [sys.executable, "-m", "pip", "install",
     "numpy", "pandas", "scipy", "scikit-learn",
     "imbalanced-learn", "matplotlib", "seaborn", "tqdm", "-q"],
    check=True,
)

import gc, re, time, warnings, hashlib, json
import numpy             as np
import pandas            as pd
import scipy.stats       as sts
import scipy.signal      as sig
import matplotlib;       matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn           as sns

from pathlib     import Path
from tqdm        import tqdm
from collections import defaultdict, deque, Counter
from dataclasses import dataclass, field
from typing      import Dict, List, Optional, Tuple

from scipy.signal import hilbert
from matplotlib.patches import Patch

warnings.filterwarnings("ignore")
np.random.seed(RANDOM_SEED)

from sklearn.preprocessing   import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics         import (accuracy_score, f1_score, precision_score,
                                      recall_score, classification_report,
                                      confusion_matrix, log_loss, roc_auc_score)
from sklearn.ensemble        import (RandomForestClassifier,
                                      GradientBoostingClassifier, IsolationForest)
from sklearn.svm             import SVC
from sklearn.linear_model    import LogisticRegression
from sklearn.neural_network  import MLPClassifier
from sklearn.naive_bayes     import GaussianNB
from sklearn.mixture         import GaussianMixture, BayesianGaussianMixture
from imblearn.over_sampling  import SMOTE

print("✓ All imports ready")


# ════════════════════════════════════════════════════════════════════════════
# SECTION 2 — CLASS LABEL DEFINITIONS
# ════════════════════════════════════════════════════════════════════════════
CLASS_NAMES = {
    0: "Background RF activities",
    1: "AR drone",
    2: "Phantom drone",
}
N_CLASSES = len(CLASS_NAMES)

FOLDER_CLASS_MAP: Dict[str, int] = {
    "background": 0,
    "ar drone":   1,
    "ar_drone":   1,
    "ardrone":    1,
    "phantom":    2,
}

BUI_CLASS_MAP: Dict[str, int] = {
    "00000": 0,
    "10000": 1, "10001": 1, "10010": 1, "10011": 1,
    "10100": 1, "10101": 1, "10110": 1,
    "11000": 2, "11001": 2, "11010": 2,
}


def folder_to_class(folder_name: str) -> Optional[int]:
    fn = folder_name.lower().strip()
    for key, cls in FOLDER_CLASS_MAP.items():
        if key in fn:
            return cls
    return None


def bui_to_class(filename: str) -> Optional[int]:
    name = Path(filename).stem
    m    = re.search(r"\d{5}", name)
    if not m:
        return None
    return BUI_CLASS_MAP.get(m.group(0))


# ════════════════════════════════════════════════════════════════════════════
# SECTION 3 — FILE DISCOVERY  (auto-detect Layout A or B)
# ════════════════════════════════════════════════════════════════════════════

def discover_files(data_dir: str) -> Dict[int, List[Path]]:
    root = Path(data_dir)
    class_files: Dict[int, List[Path]] = defaultdict(list)

    if not root.exists():
        raise FileNotFoundError(
            f"\nFolder not found: {data_dir}\n"
            "Mount Google Drive and unzip the dataset first."
        )

    # Layout A — named subfolders
    layout_a = False
    for subdir in sorted(root.iterdir()):
        if not subdir.is_dir():
            continue
        cls = folder_to_class(subdir.name)
        if cls is None:
            continue
        files = sorted(subdir.rglob("*.csv"))
        if files:
            class_files[cls].extend(files)
            layout_a = True

    if layout_a:
        print("  Layout A detected (named subfolders):")
        for c, fl in sorted(class_files.items()):
            print(f"    [{c}] {CLASS_NAMES[c]:<35} {len(fl):>4} files")
        return dict(class_files)

    # Layout B — BUI filenames
    print("  Layout A not found — trying Layout B (BUI filename mapping)…")
    for fp in sorted(root.rglob("*.csv")):
        cls = bui_to_class(fp.name)
        if cls is not None:
            class_files[cls].append(fp)

    if not class_files:
        raise RuntimeError(
            "No CSV files could be mapped to a class.\n"
            "Expected subfolders 'AR drone', 'Phantom drone', "
            "'Background RF activities'\nOR BUI-prefixed files like 10001H3.csv"
        )

    print("  Layout B detected (BUI filename mapping):")
    for c, fl in sorted(class_files.items()):
        print(f"    [{c}] {CLASS_NAMES[c]:<35} {len(fl):>4} files")
    return dict(class_files)


# ════════════════════════════════════════════════════════════════════════════
# SECTION 4 — STAGE 1: FEATURE EXTRACTION
#
#   30 features in 5 groups, designed for short windows (1024 samples):
#
#   Group A (8): Raw amplitude statistics
#     mean, std, var, min, max, range, kurtosis, skew
#     → Captures modulation depth, power level, waveform dynamics.
#       Kurtosis distinguishes impulsive signals (FHSS bursts) from smooth
#       continuous signals (background).
#
#   Group B (6): Hilbert analytic signal features
#     envelope mean/std/min/max, signal_power_dB, IQ_correlation
#     → Hilbert transform gives the analytic signal A(t)·e^{jφ(t)}.
#       Envelope = A(t) = amplitude modulation profile of the carrier.
#       IQ_corr = Pearson r between I and Q channels; each drone link
#       protocol (OcuSync, Lightbridge, WiFi, DSMx) has a distinct IQ
#       correlation signature, making this one of the most discriminative
#       single features.
#
#   Group C (8): Welch PSD spectral features
#     peak_freq_hz, bandwidth_hz, spectral_entropy, centroid, spread,
#     rolloff_85, psd_mean_dB, psd_max_dB
#     → Welch method averages multiple FFT frames → stable PSD estimate.
#       bandwidth_hz at −10 dB separates FHSS (wide) from DSSS (narrow).
#       Spectral entropy is HIGH for frequency-hopping, LOW for fixed-channel.
#       Centroid tracks the "centre of gravity" of the spectrum.
#
#   Group D (4): Instantaneous frequency statistics
#     ifreq_mean, ifreq_std, ifreq_range, ifreq_kurtosis
#     → ifreq = dφ/dt (derivative of unwrapped phase) = FM modulation rate.
#       Kurtosis of ifreq catches bursty FHSS patterns that look "spiky"
#       compared to the smooth FM of analogue video links.
#
#   Group E (4): Band energy ratios
#     energy_band1..4 (four equal-width quarters of the one-sided FFT)
#     → Scale-invariant: same drone at 10 m vs 100 m → same ratios.
#       Different drone families use different sub-bands of the 2.4 GHz
#       or 5.8 GHz ISM band, giving each a distinctive energy distribution.
# ════════════════════════════════════════════════════════════════════════════

FEATURE_NAMES: List[str] = (
    ["amp_mean", "amp_std", "amp_var", "amp_min",
     "amp_max", "amp_range", "amp_kurtosis", "amp_skew"] +
    ["env_mean", "env_std", "env_min", "env_max",
     "signal_power_db", "IQ_corr"] +
    ["peak_freq_hz", "bandwidth_hz", "spectral_entropy",
     "spectral_centroid", "spectral_spread", "spectral_rolloff_85",
     "psd_mean_db", "psd_max_db"] +
    ["ifreq_mean", "ifreq_std", "ifreq_range", "ifreq_kurtosis"] +
    ["energy_band1", "energy_band2", "energy_band3", "energy_band4"]
)
N_FEATURES = len(FEATURE_NAMES)   # 30
assert N_FEATURES == 30, f"Expected 30, got {N_FEATURES}"
FIDX = {n: i for i, n in enumerate(FEATURE_NAMES)}


FS = 10e6  # sampling frequency (Hz)
def extract_features(segment: np.ndarray, fs: float) -> np.ndarray:
    """Extract 30 RF features from a single amplitude segment."""
    amp = segment.astype(np.float64)

    # Group A — raw amplitude stats
    feat = [
        float(np.mean(amp)), float(np.std(amp)), float(np.var(amp)),
        float(np.min(amp)),  float(np.max(amp)), float(np.ptp(amp)),
        float(sts.kurtosis(amp)), float(sts.skew(amp)),
    ]

    # Group B — Hilbert envelope
    analytic = hilbert(amp)
    I = np.real(analytic)
    Q = np.imag(analytic)
    envelope = np.abs(analytic)

    power_db = float(10.0 * np.log10(np.mean(envelope ** 2) + 1e-12))

    if np.std(I) > 1e-12 and np.std(Q) > 1e-12:
        iq_corr = float(np.corrcoef(I, Q)[0, 1])
    else:
        iq_corr = 0.0

    feat += [
        float(np.mean(envelope)), float(np.std(envelope)),
        float(np.min(envelope)),  float(np.max(envelope)),
        power_db, iq_corr
    ]

    # Group C — Welch PSD
    fw, psd = sig.welch(amp, fs=fs, nperseg=256, noverlap=128, return_onesided=True)
    pa = np.abs(psd)
    pd_db = 10.0 * np.log10(pa + 1e-12)

    pk_idx = int(np.argmax(pa))

    above = fw[pd_db > pd_db[pk_idx] - 10.0]
    bw = float(above.max() - above.min()) if len(above) > 1 else 0.0

    pn = pa / (pa.sum() + 1e-12)
    entropy = float(-np.sum(pn * np.log2(pn + 1e-12)))

    s = pa.sum() + 1e-12
    cen = float(np.sum(fw * pa) / s)
    spread = float(np.sqrt(np.sum(((fw - cen) ** 2) * pa) / s))

    cs = np.cumsum(pa)
    rol_idx = min(np.searchsorted(cs, 0.85 * cs[-1]), len(fw) - 1)

    feat += [
        float(fw[pk_idx]), bw, entropy, cen, spread,
        float(fw[rol_idx]), float(np.mean(pd_db)), float(np.max(pd_db))
    ]

    # Group D — instantaneous frequency
    phase = np.unwrap(np.angle(analytic))
    ifreq = np.diff(phase)

    feat += [
        float(np.mean(ifreq)), float(np.std(ifreq)),
        float(np.ptp(ifreq)),  float(sts.kurtosis(ifreq))
    ]

    # Group E — band energy ratios
    n = len(pa)
    q = max(1, n // 4)
    tot = pa.sum() + 1e-12

    feat += [
        float(pa[0:q].sum()/tot),
        float(pa[q:2*q].sum()/tot),
        float(pa[2*q:3*q].sum()/tot),
        float(pa[3*q:].sum()/tot)
    ]

    out = np.array(feat, dtype=np.float32)
    return np.nan_to_num(out, nan=0.0, posinf=0.0, neginf=0.0)


def safe_extract(segment: np.ndarray, fs: float = FS) -> np.ndarray:
    try:
        return extract_features(segment, fs)
    except Exception:
        return np.zeros(30, dtype=np.float32)


print(f"✓ Feature pipeline ready — {N_FEATURES} features per segment")


# ════════════════════════════════════════════════════════════════════════════
# SECTION 5 — MEMORY-SAFE CHUNK READER
#
#   WHY CHUNKED READING?
#     Each DroneRF CSV ≈ 8 MB raw.  With 454 files, reading all at once
#     would spike Colab RAM to ~3.6 GB before feature extraction even starts.
#     64 MB chunks keep peak RAM under 200 MB regardless of file count.
#
#   CARRY-OVER BUFFER:
#     The last (WINDOW_SIZE−1) samples from each chunk are prepended to
#     the next chunk so no window is ever lost at a chunk boundary.
# ════════════════════════════════════════════════════════════════════════════

CHUNK_SAMPLES = CHUNK_BYTES // 4    # float32 = 4 bytes


def extract_from_file_chunked(filepath: Path,
                               quota:  int,
                               window: int = WINDOW_SIZE,
                               step:   int = STEP_SIZE) -> List[np.ndarray]:
    """Stream one CSV in memory-safe chunks, return up to `quota` segments."""
    all_segs: List[np.ndarray] = []
    carry = np.empty(0, dtype=np.float32)

    try:
        reader = pd.read_csv(
            filepath, header=None, dtype=np.float32,
            chunksize=CHUNK_SAMPLES, engine="c",
        )
        for chunk_df in reader:
            if len(all_segs) >= quota:
                break
            chunk_arr = chunk_df.values.flatten()
            buf       = np.concatenate([carry, chunk_arr]) if len(carry) else chunk_arr

            for start in range(0, len(buf) - window + 1, step):
                if len(all_segs) >= quota:
                    break
                all_segs.append(buf[start: start + window].copy())

            carry = buf[-(window - 1):].copy() if len(buf) >= window else buf.copy()
            del chunk_arr, buf
            gc.collect()

    except Exception as exc:
        print(f"    ⚠  Skipping {filepath.name}: {exc}")
        return []

    return all_segs


# ════════════════════════════════════════════════════════════════════════════
# SECTION 6 — BALANCED DATASET BUILDER
#   (DATASET PIPELINE — NOT MODIFIED PER YOUR REQUIREMENT)
#
#   Strategy:
#     per_class_quota = TARGET_TOTAL // n_classes
#     Files within each class are shuffled before sampling so all flight
#     modes (hover/fly/video/on) contribute proportionally.
#     MAX_SEGMENTS_PER_FILE caps how many windows we take from any single
#     file, preventing one dominant file from biasing the class distribution.
# ════════════════════════════════════════════════════════════════════════════

def build_dataset(data_dir: str) -> pd.DataFrame:
    print(f"\n{'='*65}")
    print("STAGE 1 — BALANCED DATASET EXTRACTION")
    print(f"  window={WINDOW_SIZE}  step={STEP_SIZE}  "
          f"target={TARGET_TOTAL:,}  fs={FS/1e6:.0f} MHz")
    print(f"{'='*65}\n")

    print("Discovering files…")
    class_files = discover_files(data_dir)
    n_cls       = len(class_files)
    quota_base  = TARGET_TOTAL // n_cls
    leftover    = TARGET_TOTAL - quota_base * n_cls

    print(f"\n  {n_cls} classes  |  {quota_base:,} segments per class")

    rng  = np.random.default_rng(RANDOM_SEED)
    rows: List[dict] = []

    for i, (cls_int, file_list) in enumerate(sorted(class_files.items())):
        cls_name = CLASS_NAMES[cls_int]
        quota    = quota_base + (1 if i < leftover else 0)
        shuffled = list(file_list)
        rng.shuffle(shuffled)

        collected = 0
        print(f"\n  [{cls_int}] {cls_name}  (quota={quota:,}):")

        for fp in shuffled:
            if collected >= quota:
                break
            segs = extract_from_file_chunked(
                fp, min(MAX_SEGMENTS_PER_FILE, quota - collected)
            )
            if not segs:
                continue

            for seg in segs:
                fv = safe_extract(seg)
                row = {"label_int": cls_int, "label_name": cls_name,
                       "source_file": fp.name}
                for fname, fval in zip(FEATURE_NAMES, fv):
                    row[fname] = fval
                rows.append(row)

            collected += len(segs)
            print(f"    {fp.name:<45}  +{len(segs):>5}  total={collected:>6}/{quota}")
            del segs;  gc.collect()

        print(f"  ✓ {cls_name}: {collected} segments extracted")

    df = (pd.DataFrame(rows)
          .sample(frac=1.0, random_state=RANDOM_SEED)
          .reset_index(drop=True))
    df.to_csv(OUTPUT_CSV, index=False)
    print(f"\n✓ Dataset saved → {OUTPUT_CSV}  ({len(df):,} rows × {N_FEATURES} features)")
    return df


# ════════════════════════════════════════════════════════════════════════════
# SECTION 7 — LOAD OR BUILD DATASET
# ════════════════════════════════════════════════════════════════════════════

_cache = Path(OUTPUT_CSV)
if _cache.exists():
    print(f"\nLoading cached dataset from {OUTPUT_CSV}…")
    df_real = pd.read_csv(_cache)
    print(f"  Loaded {len(df_real):,} rows")
else:
    df_real = build_dataset(DATA_DIR)

print("\n  Class distribution:")
for name, cnt in df_real["label_name"].value_counts().items():
    pct = cnt / len(df_real) * 100
    bar = "█" * int(pct / 2)
    print(f"    {name:<35} {cnt:>6}  ({pct:.1f}%)  {bar}")


# ════════════════════════════════════════════════════════════════════════════
# SECTION 8 — TRAIN / TEST SPLIT + SCALING + SMOTE
#
#   WHY SMOTE?
#     Even after balanced extraction, minor per-file sampling randomness
#     can leave one class with slightly fewer samples.  SMOTE synthesises
#     new training samples by interpolating between real neighbours in
#     feature space — it does NOT generate new raw RF data.
#     SMOTE is applied ONLY to the training split, never to the test split.
# ════════════════════════════════════════════════════════════════════════════

X_all = np.nan_to_num(
    df_real[FEATURE_NAMES].fillna(0).values.astype(np.float32),
    nan=0., posinf=0., neginf=0.,
)
y_all = df_real["label_int"].values.astype(np.int64)

counts        = {c: int((y_all == c).sum()) for c in np.unique(y_all)}
valid_classes = [c for c, n in counts.items() if n >= 6]
mask          = np.isin(y_all, valid_classes)
X_use, y_use  = X_all[mask], y_all[mask]

lmap            = {old: new for new, old in enumerate(sorted(valid_classes))}
y_mapped        = np.array([lmap[yi] for yi in y_use])
CLASSES_PRESENT = [CLASS_NAMES[c] for c in sorted(valid_classes)]
N_CLS           = len(CLASSES_PRESENT)
BACKGROUND_IDX  = lmap.get(0, None)

print(f"\n  Classes for training: {N_CLS}")
for i, cn in enumerate(CLASSES_PRESENT):
    print(f"    [{i}] {cn}  ({(y_mapped==i).sum()} samples)")

X_tr, X_te, y_tr, y_te = train_test_split(
    X_use, y_mapped, test_size=0.20, stratify=y_mapped, random_state=RANDOM_SEED
)

scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_tr)
X_te_s = scaler.transform(X_te)

# SMOTE — balances training set only
_, cnts  = np.unique(y_tr, return_counts=True)
k_smote  = max(1, min(3, int(min(cnts)) - 1))
sm       = SMOTE(random_state=RANDOM_SEED, k_neighbors=k_smote)
X_sm, y_sm = sm.fit_resample(X_tr_s, y_tr)

print(f"\n  After SMOTE : {X_sm.shape[0]:,} training samples")
print(f"  Test set    : {X_te_s.shape[0]:,} samples (real, held-out)")


# ════════════════════════════════════════════════════════════════════════════
# SECTION 9 — STAGE 2: DUAL BAYESIAN CLASSIFIER
#
#   WHY TWO CLASSIFIERS (RF + GNB)?
#
#   Random Forest gives a robust, discriminative P(class|x) based on
#   decision boundaries learned from many trees.  It handles nonlinear
#   feature interactions well but is NOT a generative model — it cannot
#   tell you how likely the data itself is under each class.
#
#   Gaussian Naive Bayes IS a generative Bayesian model.  It computes:
#     P(class|x) ∝ P(x|class) × P(class)
#   where P(x|class) = product of per-feature Gaussian likelihoods.
#   This gives a true Bayesian posterior AND a log-likelihood P(x|class),
#   which is used as a fourth anomaly detector.
#
#   Combined posterior = geometric mean √(P_RF × P_GNB):
#     - Geometric mean is the Bayesian model averaging formula for two
#       models with equal prior weight.
#     - It is more conservative than arithmetic mean: both models must
#       agree to produce a high combined probability.
#     - When they disagree, the combined confidence is naturally lower,
#       which routes the sample into the anomaly/uncertainty path.
#
#   Online GNB (partial_fit):
#     GaussianNB supports incremental Bayesian posterior updates.
#     Each batch update revises the class mean and variance estimates
#     using Welford's online algorithm — this is genuine Bayesian
#     updating, NOT retraining.  We use it ONLY to demonstrate and
#     plot the learning curve; the deployed model stays frozen.
#
#   CONFIDENCE GATE:
#     combined_posterior ≥ CONFIDENCE_THRESHOLD → KNOWN DRONE
#     combined_posterior <  CONFIDENCE_THRESHOLD → ANOMALY PATH
# ════════════════════════════════════════════════════════════════════════════

print(f"\n{'='*65}")
print("STAGE 2 — DUAL BAYESIAN CLASSIFIER  (one-time training, then frozen)")
print(f"{'='*65}")

# ── 2a: Random Forest ─────────────────────────────────────────────────────
t0 = time.time()
rf_clf = RandomForestClassifier(
    n_estimators     = 150,
    max_depth        = None,
    min_samples_leaf = 2,
    class_weight     = {0: 1.2, 1: 1.0, 2: 1.0},
    n_jobs           = -1,
    random_state     = RANDOM_SEED,
    oob_score        = True,
)
rf_clf.fit(X_sm, y_sm)
t_rf = round(time.time() - t0, 2)

yp_rf  = rf_clf.predict(X_te_s)
acc_rf = accuracy_score(y_te, yp_rf)
f1_rf  = f1_score(y_te, yp_rf, average="macro", zero_division=0)
print(f"\n  [RandomForest]")
print(f"    Training time  : {t_rf}s")
print(f"    OOB accuracy   : {rf_clf.oob_score_:.4f}")
print(f"    Test accuracy  : {acc_rf:.4f}")
print(f"    Test F1-macro  : {f1_rf:.4f}")

# ── 2b: Gaussian Naive Bayes (frozen) ─────────────────────────────────────
# WHY GNB AS PRIMARY CONFIDENCE ENGINE:
#   GNB produces P(class|x) via Bayes' theorem:
#     P(y=k|x) = P(x|y=k) · P(y=k) / P(x)
#   This is a proper probability from a generative model.
#   RF produces calibrated but discriminative probabilities.
#   GNB probabilities reflect how well the data fits each class model —
#   when a signal is truly unknown, GNB's P(x|class) is low for ALL
#   classes simultaneously, which drives the combined posterior down
#   and triggers the anomaly path.
t0     = time.time()
gnb_clf = GaussianNB()
gnb_clf.fit(X_sm, y_sm)
t_gnb  = round(time.time() - t0, 3)
yp_gnb = gnb_clf.predict(X_te_s)
acc_gnb= accuracy_score(y_te, yp_gnb)
f1_gnb = f1_score(y_te, yp_gnb, average="macro", zero_division=0)
print(f"\n  [GaussianNaiveBayes — Bayesian posterior]")
print(f"    Training time  : {t_gnb}s")
print(f"    Test accuracy  : {acc_gnb:.4f}")
print(f"    Test F1-macro  : {f1_gnb:.4f}")

# ── 2c: Online GNB — Bayesian incremental update (learning curve) ─────────
# WHY ONLINE GNB?
#   partial_fit() updates the per-class sufficient statistics (mean, variance)
#   with each new batch using Welford's online algorithm.  This is how
#   Bayesian inference is supposed to work: posterior = prior × likelihood,
#   iteratively updated as new data arrives.  We record the F1 after each
#   batch to show how confidence stabilises over time.
print(f"\n  [OnlineGNB — Bayesian incremental update]")
gnb_online = GaussianNB()
f1_curve: List[float] = []
t0 = time.time()
batch_size = 50
for i in range(0, len(X_sm), batch_size):
    gnb_online.partial_fit(
        X_sm[i: i + batch_size],
        y_sm[i: i + batch_size],
        classes=np.arange(N_CLS),
    )
    yp_o = gnb_online.predict(X_te_s)
    f1_curve.append(round(
        f1_score(y_te, yp_o, average="macro", zero_division=0), 4
    ))
t_online = round(time.time() - t0, 2)
print(f"    {len(f1_curve)} batches  |  final F1={f1_curve[-1]:.4f}  |  {t_online}s")
print(f"    Learning curve: {f1_curve[:4]} … {f1_curve[-3:]}")

# Feature importances
importances = rf_clf.feature_importances_
top10       = np.argsort(importances)[::-1][:10]
print(f"\n  Top 10 discriminative RF features:")
for rank, idx in enumerate(top10, 1):
    print(f"    {rank:>2}. {FEATURE_NAMES[idx]:<25} {importances[idx]:.4f}")

print("\n  *** Both classifiers are now FROZEN — no further training will occur ***")


# ════════════════════════════════════════════════════════════════════════════
# SECTION 10 — SUPPLEMENTARY MODEL BENCHMARK
#   5 additional classifiers benchmarked for comparison.
#   None of these replace the dual RF+GNB system.
# ════════════════════════════════════════════════════════════════════════════

print(f"\n{'='*65}")
print("SUPPLEMENTARY BENCHMARK — 5 additional classifiers")
print(f"{'='*65}")

bench_results: Dict[str, dict] = {}


def run_benchmark(name: str, model, cv: bool = True) -> None:
    t0  = time.time()
    model.fit(X_sm, y_sm)
    tt  = round(time.time() - t0, 3)
    yp  = model.predict(X_te_s)
    acc = round(accuracy_score(y_te, yp), 4)
    f1  = round(f1_score(y_te, yp, average="macro", zero_division=0), 4)
    prec= round(precision_score(y_te, yp, average="macro", zero_division=0), 4)
    rec = round(recall_score(y_te, yp, average="macro", zero_division=0), 4)
    ll  = (round(log_loss(y_te, model.predict_proba(X_te_s)), 4)
           if hasattr(model, "predict_proba") else None)
    cvf = None
    if cv:
        cvf = round(cross_val_score(
            model, X_sm, y_sm,
            cv=StratifiedKFold(3, shuffle=True, random_state=RANDOM_SEED),
            scoring="f1_macro", n_jobs=-1,
        ).mean(), 4)
    bench_results[name] = {
        "accuracy": acc, "f1_macro": f1, "precision": prec,
        "recall": rec, "log_loss": ll, "cv_f1": cvf, "train_s": tt,
    }
    print(f"  {name:<30} acc={acc:.4f}  f1={f1:.4f}  t={tt}s  cv={cvf}")


run_benchmark("GradientBoosting",
    GradientBoostingClassifier(n_estimators=100, learning_rate=0.1,
                                max_depth=4, random_state=RANDOM_SEED))
run_benchmark("SVM_RBF",
    SVC(kernel="rbf", C=5, gamma="scale", class_weight="balanced",
        probability=True, random_state=RANDOM_SEED))
run_benchmark("LogisticRegression",
    LogisticRegression(C=1.0, class_weight="balanced",
                        max_iter=500, random_state=RANDOM_SEED))
run_benchmark("MLP_NeuralNet",
    MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=200,
                   early_stopping=True, random_state=RANDOM_SEED))

# Dirichlet Process GMM — non-parametric Bayesian model
# WHY DP-GMM?
#   Standard GMM requires you to specify the number of clusters.
#   Dirichlet Process GMM infers the number of clusters from the data.
#   weight_concentration_prior=0.01 encourages sparsity — few active clusters.
#   A new drone's log-likelihood under the trained DP-GMM is LOW because
#   none of the inferred clusters match it.  This makes DP-GMM an excellent
#   secondary novelty detector alongside the main anomaly ensemble.
print(f"\n  Training Dirichlet Process GMM (non-parametric Bayesian)…")
t0 = time.time()
dp_gmm = BayesianGaussianMixture(
    n_components=min(20, N_CLS * 4),
    covariance_type="full",
    weight_concentration_prior_type="dirichlet_process",
    weight_concentration_prior=0.01,
    max_iter=300,
    random_state=RANDOM_SEED,
)
dp_gmm.fit(X_sm)
t_dp = round(time.time() - t0, 2)
active_clusters = int((dp_gmm.weights_ > 0.01).sum())
# Map DP-GMM clusters to class labels via majority vote
comp_labels = dp_gmm.predict(X_sm)
c2c = {int(c): int(np.bincount(y_sm[comp_labels == c]).argmax())
       for c in np.unique(comp_labels)}
yp_dp  = np.array([c2c.get(int(c), -1) for c in dp_gmm.predict(X_te_s)])
valid  = yp_dp >= 0
acc_dp = accuracy_score(y_te[valid], yp_dp[valid]) if valid.sum() > 0 else 0.0
f1_dp  = f1_score(y_te[valid], yp_dp[valid], average="macro", zero_division=0) \
         if valid.sum() > 0 else 0.0
bench_results["DirichletProcess_GMM"] = {
    "accuracy": round(acc_dp, 4), "f1_macro": round(f1_dp, 4),
    "precision": None, "recall": None, "log_loss": None,
    "cv_f1": None, "train_s": t_dp,
}
print(f"  {'DirichletProcess_GMM':<30} acc={acc_dp:.4f}  f1={f1_dp:.4f}  "
      f"t={t_dp}s  active_clusters={active_clusters}")

bench_df = pd.DataFrame(
    [{"Model": k, **v} for k, v in bench_results.items()]
).sort_values("f1_macro", ascending=False)
bench_df.to_csv("benchmark_results.csv", index=False)
print(f"\n  ✓ Benchmark saved → benchmark_results.csv")
print(f"\nFull classification report (RandomForest on test set):")
print(classification_report(y_te, yp_rf, target_names=CLASSES_PRESENT, zero_division=0))


# ════════════════════════════════════════════════════════════════════════════
# SECTION 11 — STAGE 3: ANOMALY DETECTION MODULE  (4 detectors, all frozen)
#
#   WHY FOUR DETECTORS?
#   No single detector catches every kind of unknown signal:
#
#   MahalanobisDetector (weight 0.35):
#     Measures class-conditional distance accounting for feature
#     correlations via the precision matrix Σk⁻¹.
#     A signal can be near the marginal mean of every feature but still
#     be anomalous if the *joint* feature pattern violates the covariance
#     structure.  A global Mahalanobis would miss this; per-class does not.
#     Best at: signals that look "close" in Euclidean space but have
#     wrong feature correlations (e.g., wrong IQ_corr for that power level).
#
#   GMMDetector (weight 0.30):
#     Fits one GMM per class — each class has its own density model.
#     anomaly_score = −max_k log P_k(x)
#     Per-class prevents unknown signals from hiding between class clusters
#     (a failure mode of a single global GMM).
#     Best at: signals that fall in the "gap" between class clusters.
#
#   IsoForestDetector (weight 0.25):
#     Builds random trees that recursively partition the feature space.
#     Anomalous points are isolated with fewer splits (shorter path length).
#     Makes NO distributional assumption → catches any geometric outlier.
#     Best at: encrypted/spread-spectrum signals with unusual feature topology.
#
#   GNBDetector (weight 0.10):
#     Uses the log-likelihood from the trained GNB: −max_k log P(x|class=k).
#     When a signal is truly novel, it has low likelihood under ALL class
#     models simultaneously.  Lower weight because GNB assumes feature
#     independence which may not hold for correlated RF features.
#     Best at: signals that violate multiple marginal Gaussians at once.
# ════════════════════════════════════════════════════════════════════════════

class MahalanobisDetector:
    """Class-conditional Mahalanobis distance anomaly detector."""

    def fit(self, X: np.ndarray, y: np.ndarray) -> "MahalanobisDetector":
        self.params: Dict[int, Tuple[np.ndarray, np.ndarray]] = {}
        for c in np.unique(y):
            Xc   = X[y == c]
            mu   = np.mean(Xc, axis=0)
            cov  = np.cov(Xc, rowvar=False) + np.eye(Xc.shape[1]) * 1e-3
            prec = np.linalg.inv(cov)
            self.params[c] = (mu, prec)
        self.threshold = float(np.percentile(self.score(X), 99))
        return self

    def score(self, X: np.ndarray) -> np.ndarray:
        dists = [
            np.sqrt(np.maximum(
                np.einsum("ni,ij,nj->n", X - mu, prec, X - mu), 0.0
            ))
            for mu, prec in self.params.values()
        ]
        return np.min(np.stack(dists, axis=1), axis=1)


class GMMDetector:
    """Per-class GMM density anomaly detector."""

    def fit(self, X: np.ndarray, y: np.ndarray) -> "GMMDetector":
        self.gmms: Dict[int, GaussianMixture] = {}
        for c in np.unique(y):
            Xc = X[y == c]
            nc = min(3, max(1, len(Xc) // 10))
            self.gmms[c] = GaussianMixture(
                n_components=nc, covariance_type="diag",
                random_state=RANDOM_SEED, reg_covar=1e-4, max_iter=200,
            ).fit(Xc)
        self.threshold = float(np.percentile(self.score(X), 99))
        return self

    def _max_ll(self, X: np.ndarray) -> np.ndarray:
        ll = np.stack([g.score_samples(X) for g in self.gmms.values()], axis=1)
        return ll.max(axis=1)

    def score(self, X: np.ndarray) -> np.ndarray:
        return -self._max_ll(X)


class IsoForestDetector:
    """Isolation Forest geometric anomaly detector."""

    def fit(self, X: np.ndarray, y=None) -> "IsoForestDetector":
        self.model = IsolationForest(
            n_estimators=200, contamination=0.02, random_state=RANDOM_SEED
        ).fit(X)
        self.threshold = float(
            np.percentile(-self.model.score_samples(X), 99)
        )
        return self

    def score(self, X: np.ndarray) -> np.ndarray:
        return -self.model.score_samples(X)


class GNBLikelihoodDetector:
    """Gaussian NB log-likelihood anomaly detector."""

    def fit(self, X: np.ndarray, y: np.ndarray) -> "GNBLikelihoodDetector":
        # Reuse the already-trained gnb_clf — no extra computation needed
        self.gnb = gnb_clf
        self.threshold = float(np.percentile(self.score(X), 99))
        return self

    def score(self, X: np.ndarray) -> np.ndarray:
        # predict_log_proba returns log P(class|x) ∝ log P(x|class) + log P(class)
        # The maximum is highest for the best-matching class.
        # Negated → higher = more anomalous (low likelihood under all classes).
        return -self.gnb.predict_log_proba(X).max(axis=1)


print(f"\n{'='*65}")
print("STAGE 3 — ANOMALY DETECTION  (one-time training)")
print(f"{'='*65}")

t0 = time.time(); det_mahal = MahalanobisDetector().fit(X_sm, y_sm)
print(f"  Mahalanobis  ✓  ({time.time()-t0:.1f}s)  threshold={det_mahal.threshold:.3f}")

t0 = time.time(); det_gmm = GMMDetector().fit(X_sm, y_sm)
print(f"  Per-Class GMM✓  ({time.time()-t0:.1f}s)")

t0 = time.time(); det_iso = IsoForestDetector().fit(X_sm)
print(f"  IsoForest    ✓  ({time.time()-t0:.1f}s)")

t0 = time.time(); det_gnb_ll = GNBLikelihoodDetector().fit(X_sm, y_sm)
print(f"  GNB-LL       ✓  ({time.time()-t0:.1f}s)")

print("\n  *** All 4 anomaly detectors FROZEN ***")


# ════════════════════════════════════════════════════════════════════════════

# Calibrate normalisation parameters from training data
_norm_params: Dict[str, Tuple[float, float]] = {}
for _name, _det in [("mahal", det_mahal), ("gmm", det_gmm),
                     ("isoforest", det_iso), ("gnb", det_gnb_ll)]:
    _s = _det.score(X_sm)
    _norm_params[_name] = (float(np.percentile(_s, 1)),
                            float(np.percentile(_s, 99)))


def _norm_score(name: str, raw: np.ndarray) -> np.ndarray:
    lo, hi = _norm_params[name]
    return np.clip((raw - lo) / (hi - lo + 1e-12), 0.0, 1.0)


def compute_threat_score(X_sc: np.ndarray) -> np.ndarray:
    """Ensemble anomaly score ∈ [0,1].  Higher = more anomalous."""
    return (ANOMALY_WEIGHTS["mahal"]     * _norm_score("mahal",     det_mahal.score(X_sc))   +
            ANOMALY_WEIGHTS["gmm"]       * _norm_score("gmm",       det_gmm.score(X_sc))     +
            ANOMALY_WEIGHTS["isoforest"] * _norm_score("isoforest", det_iso.score(X_sc))     +
            ANOMALY_WEIGHTS["gnb"]       * _norm_score("gnb",       det_gnb_ll.score(X_sc)))


def bayesian_confidence_breakdown(
    X_sc:    np.ndarray,      # scaled feature vector (1, N_FEATURES)
    ts:      float,           # pre-computed threat score
) -> dict:
    """
    Compute the full Bayesian confidence breakdown for one prediction.

    Returns
    -------
    dict with keys:
      winner                — predicted class name
      posterior_probs       — combined P(class|x) per class (RF × GNB geom. mean)
      rf_probs              — RF-only P(class|x)
      gnb_probs             — GNB-only P(class|x)  [true Bayesian posterior]
      max_confidence        — max combined posterior
      predictive_entropy    — H[y|x] / log(N) ∈ [0,1]  total uncertainty
      epistemic_uncertainty — model knowledge gap (anomaly-driven) ∈ [0,1]
      aleatoric_uncertainty — irreducible signal ambiguity ∈ [0,1]
      margin                — P(winner) − P(second)
      calibrated_confidence — margin-adjusted final confidence ∈ [0,1]
      n_credible_classes    — classes needed to cover 95% posterior mass
      is_novel              — bool: signal likely not from training distribution
      gnb_log_likelihood    — per-class log P(x|class) from GNB
    """
    eps = 1e-12
    n   = N_CLS

    # RF probabilities (discriminative posterior)
    rf_p  = rf_clf.predict_proba(X_sc)[0].astype(np.float64)
    rf_p  = rf_p + eps;  rf_p  /= rf_p.sum()

    # GNB probabilities (generative Bayesian posterior)
    gnb_p = gnb_clf.predict_proba(X_sc)[0].astype(np.float64)
    gnb_p = gnb_p + eps;  gnb_p /= gnb_p.sum()

    # GNB log-likelihood per class: log P(x|class)
    # predict_log_proba returns log P(class|x), but we want P(x|class).
    # Using: log P(x|class) = log P(class|x) − log P(class) + const
    # For simplicity we use predict_log_proba max as a novelty signal.
    gnb_log_ll = gnb_clf.predict_log_proba(X_sc)[0]

    # Combined posterior: geometric mean (Bayesian model averaging)
    combined = np.sqrt(rf_p * gnb_p)
    combined /= combined.sum()

    winner_idx = int(np.argmax(combined))
    sorted_c   = np.sort(combined)[::-1]

    # Total predictive entropy H[y|x] normalised to [0,1]
    pred_H    = float(-np.sum(combined * np.log(combined + eps)))
    norm_H    = float(pred_H / (np.log(n) + eps))

    # IsoForest normalised score (best single proxy for epistemic uncertainty)
    iso_norm  = float(_norm_score("isoforest", det_iso.score(X_sc))[0])

    # Epistemic uncertainty: "does the model know anything about this signal?"
    epistemic = float(np.clip(0.5 * iso_norm + 0.5 * norm_H, 0.0, 1.0))

    # Aleatoric uncertainty: "is the signal inherently ambiguous?"
    aleatoric = float(np.clip(norm_H * (1.0 - ts), 0.0, 1.0))

    # Margin between top-2 classes
    margin = float(sorted_c[0] - sorted_c[1]) if n > 1 else 1.0

    # Calibrated confidence: rewards decisive predictions, penalises marginal ones
    calibrated = float(sorted_c[0] * (1.0 + margin) / 2.0)

    # Credible interval: classes needed to cover 95% of probability mass
    cum        = np.cumsum(np.sort(combined)[::-1])
    n_credible = int(np.searchsorted(cum, 0.95)) + 1

    # Novel signal detection heuristic
    is_novel = bool(ts > 0.5 or norm_H > 0.85 or margin < 0.10)

    return {
        "winner":                CLASSES_PRESENT[winner_idx],
        "posterior_probs":       {CLASSES_PRESENT[i]: round(float(combined[i]), 4)
                                   for i in range(n)},
        "rf_probs":              {CLASSES_PRESENT[i]: round(float(rf_p[i]), 4)
                                   for i in range(n)},
        "gnb_probs":             {CLASSES_PRESENT[i]: round(float(gnb_p[i]), 4)
                                   for i in range(n)},
        "max_confidence":        round(float(sorted_c[0]), 4),
        "predictive_entropy":    round(norm_H, 4),
        "epistemic_uncertainty": round(epistemic, 4),
        "aleatoric_uncertainty": round(aleatoric, 4),
        "margin":                round(margin, 4),
        "calibrated_confidence": round(calibrated, 4),
        "n_credible_classes":    n_credible,
        "threat_score":          round(float(ts), 4),
        "is_novel":              is_novel,
        "gnb_log_likelihood":    {CLASSES_PRESENT[i]: round(float(gnb_log_ll[i]), 3)
                                   for i in range(n)},
    }


# Calibrate global threat threshold
_train_ts        = compute_threat_score(X_sm)
THREAT_THRESHOLD = float(np.percentile(_train_ts, 97))
print(f"\n  Threat score threshold : {THREAT_THRESHOLD:.4f}  "
      f"(97th-percentile of training ensemble scores)")
print("  *** Threat scoring system ready ***")


FINGERPRINT_FEAT_INDICES: List[int] = [
    FIDX["IQ_corr"], FIDX["signal_power_db"], FIDX["spectral_entropy"],
    FIDX["bandwidth_hz"], FIDX["spectral_centroid"],
    FIDX["env_mean"], FIDX["energy_band1"],
]


def emitter_hash(fv: np.ndarray, n_bins: int = 20) -> str:
    """MD5 of quantised fingerprint features → 12-char hex ID."""
    fp  = fv[FINGERPRINT_FEAT_INDICES]
    qfp = np.round(fp * n_bins).astype(np.int32)
    return hashlib.md5(qfp.tobytes()).hexdigest()[:12]


@dataclass
class EmitterRecord:
    """Tracks all observations of a single RF emitter (identified by hash)."""
    emitter_id:      str
    feature_history: deque = field(default_factory=lambda: deque(maxlen=50))
    first_seen:      float = field(default_factory=time.time)
    last_seen:       float = field(default_factory=time.time)
    seen_count:      int   = 0
    threat_scores:   List[float] = field(default_factory=list)
    trust_score:     float = 0.0
    promoted:        bool  = False

    def update(self, fv: np.ndarray, ts: float) -> None:
        self.feature_history.append(fv.copy())
        self.last_seen    = time.time()
        self.seen_count  += 1
        self.threat_scores.append(float(ts))

    @property
    def mean_features(self) -> np.ndarray:
        return np.mean(np.stack(list(self.feature_history)), axis=0)

    @property
    def feature_variance(self) -> float:
        if len(self.feature_history) < 2:
            return 1.0
        arr = np.stack(list(self.feature_history), axis=0)
        return float(np.mean(np.var(arr, axis=0)))

    @property
    def mean_threat(self) -> float:
        return float(np.mean(self.threat_scores)) if self.threat_scores else 1.0

    def compute_trust(self) -> float:
        """Harmonic mean of obs_trust, stab_trust, safe_trust."""
        obs_t  = float(1.0 / (1.0 + np.exp(
            -(self.seen_count - TRUST_MIN_OBSERVATIONS) / 3.0
        )))
        stab_t = float(max(0.0, 1.0 - self.feature_variance / (TRUST_MAX_VARIANCE + 1e-9)))
        safe_t = float(max(0.0, 1.0 - self.mean_threat))
        vals   = [obs_t, stab_t, safe_t]
        # Harmonic mean — any zero component drives result to zero
        hm     = float(len(vals) / sum(1.0 / (v + 1e-9) for v in vals))
        self.trust_score = float(np.clip(hm, 0.0, 1.0))
        return self.trust_score

    def is_trustworthy(self) -> bool:
        return (self.seen_count        >= TRUST_MIN_OBSERVATIONS and
                self.feature_variance  <= TRUST_MAX_VARIANCE      and
                self.mean_threat        < HIGH_THREAT_THRESHOLD)


class TemporalTracker:
    """In-memory registry of all unknown RF emitters observed so far."""

    def __init__(self) -> None:
        self.registry:  Dict[str, EmitterRecord] = {}
        self.total_obs: int = 0

    def observe(self, fv: np.ndarray, ts: float) -> EmitterRecord:
        eid = emitter_hash(fv)
        if eid not in self.registry:
            self.registry[eid] = EmitterRecord(emitter_id=eid)
        rec = self.registry[eid]
        rec.update(fv, ts)
        rec.compute_trust()
        self.total_obs += 1
        return rec

    def summary(self) -> str:
        n   = len(self.registry)
        nt  = sum(1 for r in self.registry.values() if r.is_trustworthy())
        nth = sum(1 for r in self.registry.values()
                  if r.mean_threat >= HIGH_THREAT_THRESHOLD)
        return (f"Tracker: {n} emitters  |  "
                f"trustworthy={nt}  threat={nth}  monitor={n-nt-nth}")


temporal_tracker = TemporalTracker()
print("\n✓ Temporal tracker initialised")


# ════════════════════════════════════════════════════════════════════════════
# SECTION 14 — STAGE 6: TRUSTED RF FINGERPRINT DATABASE
#
#   Two JSON-persistent stores:
#     trusted_db     — emitters confirmed safe after temporal vetting
#     suspicious_db  — emitters with persistently high threat scores
#
#   Each entry stores the MEAN feature vector of the emitter (not raw samples).
#   Memory: 30 × 4 bytes = 120 bytes per entry (vs 50 × 120 = 6 KB for history).
#
#   COSINE SIMILARITY MATCHING:
#     cos(θ) = (a · b) / (||a|| · ||b||)
#     WHY COSINE (not Euclidean)?
#       The same drone at 10 m vs 100 m has very different raw power levels
#       (signal_power_dB, amp_mean, env_mean all shift significantly).
#       Euclidean distance would call these "different drones".
#       Cosine similarity measures the ANGLE between feature vectors, which
#       is invariant to scaling.  The spectral SHAPE (entropy, centroid,
#       band ratios, IQ_corr) is preserved across range — cosine captures
#       this.  Two observations of the same drone at different ranges will
#       have cosine similarity close to 1.0.
# ════════════════════════════════════════════════════════════════════════════

def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    a = a.flatten().astype(np.float64)
    b = b.flatten().astype(np.float64)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-12))


class FingerprintDatabase:
    """JSON-backed RF emitter fingerprint store."""

    def __init__(self, path: str) -> None:
        self.path       = path
        self.trusted:    Dict[str, dict] = {}
        self.suspicious: Dict[str, dict] = {}
        self._load()

    def _load(self) -> None:
        if Path(self.path).exists():
            try:
                data            = json.load(open(self.path))
                self.trusted    = data.get("trusted",    {})
                self.suspicious = data.get("suspicious", {})
                print(f"  Loaded RF DB: "
                      f"{len(self.trusted)} trusted, "
                      f"{len(self.suspicious)} suspicious")
            except Exception:
                print("  RF DB corrupted — starting fresh")
        else:
            print("  RF DB: starting fresh")

    def save(self) -> None:
        json.dump(
            {"trusted": self.trusted, "suspicious": self.suspicious},
            open(self.path, "w"), indent=2,
        )

    def match(self, fv: np.ndarray) -> Tuple[Optional[str], float, str]:
        """
        Search both stores for the best cosine match.
        Returns (emitter_id, similarity, store_name) or (None, 0, '').
        """
        best_sim, best_id, best_store = -1.0, None, ""
        for store_name, db in [("trusted", self.trusted),
                                 ("suspicious", self.suspicious)]:
            for eid, rec in db.items():
                sim = cosine_similarity(fv, np.array(rec["fingerprint"]))
                if sim > best_sim:
                    best_sim, best_id, best_store = sim, eid, store_name
        return best_id, float(best_sim), best_store

    def add_trusted(self, eid: str, fv: np.ndarray,
                     seen_count: int = 0) -> None:
        if eid not in self.trusted:
            label = f"SAFE_UNKNOWN_{len(self.trusted)+1:03d}"
            self.trusted[eid] = {
                "fingerprint": fv.tolist(),
                "label":       label,
                "seen_count":  seen_count,
                "added_at":    time.time(),
            }
            self.save()
            print(f"  ✅  PROMOTED → {label}  (seen={seen_count})")

    def add_suspicious(self, eid: str, fv: np.ndarray,
                        seen_count: int = 0) -> None:
        if eid not in self.suspicious:
            label = f"THREAT_{len(self.suspicious)+1:03d}"
            self.suspicious[eid] = {
                "fingerprint": fv.tolist(),
                "label":       label,
                "seen_count":  seen_count,
                "added_at":    time.time(),
            }
        else:
            self.suspicious[eid]["seen_count"] = seen_count
        self.save()

    def summary(self) -> str:
        return (f"RF Database: {len(self.trusted)} trusted  "
                f"| {len(self.suspicious)} suspicious")


fp_db = FingerprintDatabase(DB_PATH)
print(f"✓ {fp_db.summary()}")


# ════════════════════════════════════════════════════════════════════════════
# SECTION 15 — STAGE 7: FINAL DECISION LOGIC
#
#   classify_signal() is the single entry point for any RF feature vector.
#
#   DECISION FLOW (in strict order — order matters!):
#
#   ① RF CLASSIFIER CONFIDENCE CHECK
#      combined posterior ≥ CONFIDENCE_THRESHOLD?
#        Yes → FRIENDLY_DRONE or BACKGROUND
#        No  → continue to anomaly path
#      WHY CHECK THIS FIRST: Known drones should be identified fast
#      without running all 4 anomaly detectors.
#
#   ② TRUSTED DB SIMILARITY CHECK
#      cosine(fv, trusted_fingerprints) ≥ SIMILARITY_THRESHOLD?
#        Yes → TRUSTED_NEW_DRONE (previously vetted new emitter)
#        No  → continue
#      WHY BEFORE THREAT CHECK: A previously approved drone should not
#      be re-flagged as a threat just because the classifier is unsure.
#
#   ③ THREAT SCORE CHECK
#      ensemble threat score ≥ THREAT_THRESHOLD?
#        seen_count ≥ 3 → CONFIRMED_THREAT (persistent high-anomaly)
#        else           → POTENTIAL_THREAT (first high-anomaly sighting)
#        Add to suspicious DB.
#      WHY COUNT THRESHOLD: A single anomalous reading could be noise.
#      Requiring 3+ sightings prevents single-packet false alarms.
#
#   ④ TEMPORAL TRUST PROMOTION
#      is_trustworthy() == True AND not yet promoted?
#        → SAFE_NEW_DRONE, add to trusted DB
#      Already in trusted DB?
#        → SAFE_NEW_DRONE (label = DB label)
#      Otherwise:
#        → UNKNOWN_MONITOR (accumulating evidence)
#
#   OUTPUT LABELS:
#     🟢 FRIENDLY_DRONE    — high-confidence known drone
#     ⚪ BACKGROUND        — no drone RF detected
#     🔷 TRUSTED_NEW_DRONE — matches a previously trusted emitter (cosine sim)
#     🔴 POTENTIAL_THREAT  — high anomaly, first/second sighting
#     🚨 CONFIRMED_THREAT  — high anomaly, seen 3+ times
#     🔵 SAFE_NEW_DRONE    — stable, frequent, low-threat → promoted to DB
#     🟡 UNKNOWN_MONITOR   — insufficient evidence yet
# ════════════════════════════════════════════════════════════════════════════

ICONS = {
    "FRIENDLY_DRONE":    "🟢",
    "BACKGROUND":        "⚪",
    "POTENTIAL_THREAT":  "🔴",
    "CONFIRMED_THREAT":  "🚨",
    "SAFE_NEW_DRONE":    "🔵",
    "TRUSTED_NEW_DRONE": "🔷",
    "UNKNOWN_MONITOR":   "🟡",
}


def classify_signal(fv_raw: np.ndarray,
                    return_bayes: bool = True) -> dict:
    """
    Full 7-stage classification of a single raw feature vector.

    Parameters
    ----------
    fv_raw       : 30-d feature vector from extract_features()  (UNSCALED)
    return_bayes : if True, include full Bayesian confidence breakdown

    Returns
    -------
    dict with:
        label                  — final decision string
        bayesian               — full Bayesian confidence breakdown (if requested)
        emitter_id             — 12-char hash identifying the RF emitter
        trust_score            — temporal trust ∈ [0,1] (0 if first sighting)
        promoted               — True if newly added to trusted DB
    """
    fv_raw = np.nan_to_num(fv_raw.astype(np.float32), nan=0., posinf=0., neginf=0.)
    X_sc   = scaler.transform(fv_raw.reshape(1, -1))
    eid    = emitter_hash(fv_raw)
    ts     = float(compute_threat_score(X_sc)[0])

    # Build Bayesian confidence report
    bayes = bayesian_confidence_breakdown(X_sc, ts) if return_bayes else {}

    result = {
        "label":        None,
        "bayesian":     bayes,
        "emitter_id":   eid,
        "trust_score":  0.0,
        "promoted":     False,
    }

    # ① Known drone path (confidence gate)
    conf = bayes.get("calibrated_confidence", 0.0) if bayes else 0.0
    if not bayes:
        rf_p    = rf_clf.predict_proba(X_sc)[0]
        gnb_p   = gnb_clf.predict_proba(X_sc)[0]
        combined= np.sqrt(rf_p * gnb_p + 1e-12)
        combined/= combined.sum()
        conf    = float(combined.max())

    if conf >= CONFIDENCE_THRESHOLD:
        winner = bayes["winner"] if bayes else CLASSES_PRESENT[int(np.argmax(combined))]
        result["label"] = ("BACKGROUND"
                            if winner == CLASS_NAMES.get(0)
                            else "FRIENDLY_DRONE")
        return result

    # ② Trusted DB check (previously vetted new emitter)
    match_id, sim, store = fp_db.match(fv_raw)
    if sim >= SIMILARITY_THRESHOLD and store == "trusted":
        fp_label = fp_db.trusted[match_id]["label"]
        result["label"] = "TRUSTED_NEW_DRONE"
        if bayes:
            bayes["db_match"] = fp_label
            bayes["db_similarity"] = round(float(sim), 4)
        return result

    # ③ Threat score check (anomaly path)
    rec = temporal_tracker.observe(fv_raw, ts)
    result["trust_score"] = float(rec.trust_score)
    result["emitter_id"]  = rec.emitter_id

    if ts >= THREAT_THRESHOLD or rec.mean_threat >= HIGH_THREAT_THRESHOLD:
        fp_db.add_suspicious(rec.emitter_id, rec.mean_features, rec.seen_count)
        result["label"] = ("CONFIRMED_THREAT"
                            if rec.seen_count >= 3
                            else "POTENTIAL_THREAT")
        return result

    # ④ Temporal trust promotion
    if rec.is_trustworthy() and not rec.promoted:
        fp_db.add_trusted(rec.emitter_id, rec.mean_features, rec.seen_count)
        result["promoted"] = True

    if rec.promoted or rec.emitter_id in fp_db.trusted:
        result["label"] = "SAFE_NEW_DRONE"
    else:
        result["label"] = "UNKNOWN_MONITOR"

    return result


print("✓ classify_signal() ready")


# ════════════════════════════════════════════════════════════════════════════
# SECTION 16 — SYNTHETIC THREAT & SAFE DRONE SIMULATION
#
#   Four synthetic profiles placed OUTSIDE the training distribution
#   to validate the full pipeline end-to-end without additional hardware.
#
#   Threat profiles  (DJI_Neo, Autel_EVO3):
#     Large offset from training mean + HIGH per-sample noise.
#     → Erratic feature vectors → high threat score → CONFIRMED_THREAT
#
#   Safe profiles  (Harmless_Surveyor, Delivery_Bot):
#     Moderate offset + VERY LOW noise → stable consistent features.
#     → Accumulates trust over observations → promoted to SAFE_NEW_DRONE
#
#   WHY SIMULATE?  The DroneRF dataset contains only 3 known drone types
#   (AR, Phantom, Background).  Any real deployment will encounter drones
#   not in the training set.  Simulation with controlled profiles verifies
#   that the temporal and DB stages work correctly before field deployment.
# ════════════════════════════════════════════════════════════════════════════

SYNTHETIC_PROFILES = {
    "DJI_Neo_Threat": {
        "power": -45.0, "entropy": 2.8, "bw": 20e6,
        "noise_scale": 0.5, "is_threat": True, "seed": 3001,
        "note": "OcuSync 3 / 5.8 GHz — erratic hopping, outside training",
    },
    "Autel_EVO3_Threat": {
        "power": -52.0, "entropy": 3.5, "bw": 12e6,
        "noise_scale": 0.5, "is_threat": True, "seed": 3002,
        "note": "Aggressive FHSS, 2024 model — high spectral entropy",
    },
    "Harmless_Surveyor": {
        "power": -60.0, "entropy": 1.8, "bw": 5e6,
        "noise_scale": 0.05, "is_threat": False, "seed": 3003,
        "note": "Stable surveyor — consistent narrowband, low power",
    },
    "Delivery_Bot": {
        "power": -58.0, "entropy": 2.0, "bw": 4e6,
        "noise_scale": 0.05, "is_threat": False, "seed": 3004,
        "note": "Urban delivery drone — stable, repeatedly observed",
    },
}
N_OBS = TRUST_MIN_OBSERVATIONS + 5


def generate_synthetic_obs(prof: dict, n: int = N_OBS) -> List[np.ndarray]:
    """Generate n synthetic feature vectors for a given drone profile."""
    rng  = np.random.default_rng(prof["seed"])
    base = rng.standard_normal(N_FEATURES) * 0.4
    base[FIDX["signal_power_db"]]  = prof["power"]   + rng.standard_normal() * 2
    base[FIDX["spectral_entropy"]] = prof["entropy"]  + rng.standard_normal() * 0.15
    base[FIDX["bandwidth_hz"]]     = prof["bw"]       + rng.standard_normal() * 5e5
    base[FIDX["amp_mean"]]         = rng.uniform(3.5, 5.5)
    base[FIDX["env_mean"]]         = rng.uniform(3.5, 5.5)
    return [
        (base + rng.standard_normal(N_FEATURES) * prof["noise_scale"]).astype(np.float32)
        for _ in range(n)
    ]


print(f"\n{'='*65}")
print("SYNTHETIC DRONE SIMULATION")
print(f"  Each profile observed {N_OBS} times through the full pipeline")
print(f"{'='*65}")

sim_results: Dict[str, List[dict]] = {}

for drone_name, prof in SYNTHETIC_PROFILES.items():
    print(f"\n── {drone_name}  [{prof['note']}]")
    obs       = generate_synthetic_obs(prof)
    decisions = []

    for step, fv in enumerate(obs, 1):
        dec  = classify_signal(fv, return_bayes=True)
        decisions.append(dec)
        b    = dec["bayesian"]
        icon = ICONS.get(dec["label"], "❓")
        promo= " ← PROMOTED" if dec.get("promoted") else ""
        print(
            f"  t={step:>2}  {icon} {dec['label']:<22}  "
            f"calibrated_conf={b.get('calibrated_confidence',0):.3f}  "
            f"ts={b.get('threat_score',0):.3f}  "
            f"epistemic={b.get('epistemic_uncertainty',0):.3f}  "
            f"trust={dec['trust_score']:.3f}"
            f"{promo}"
        )

    sim_results[drone_name] = decisions
    final = decisions[-1]
    print(f"  FINAL: {ICONS.get(final['label'],'?')} {final['label']}  "
          f"calibrated_conf={final['bayesian'].get('calibrated_confidence',0):.3f}")

print(f"\n{temporal_tracker.summary()}")
print(fp_db.summary())


# ════════════════════════════════════════════════════════════════════════════
# SECTION 17 — REAL-TIME DETECTION LOOP
#
#   Processes a DroneRF CSV file one sliding window at a time, printing
#   the full Bayesian confidence breakdown per segment.
#   In production: replace pd.read_csv with a GNU Radio / RTL-SDR buffer.
# ════════════════════════════════════════════════════════════════════════════

def realtime_detection_loop(csv_path: str,
                             max_segments: int = 15,
                             verbose: bool = True) -> List[dict]:
    """Classify a DroneRF CSV file one window at a time with full Bayesian output."""
    try:
        amp = pd.read_csv(
            csv_path, header=None, dtype=np.float32
        ).values.flatten()
    except Exception as e:
        print(f"ERROR reading {csv_path}: {e}")
        return []

    decisions = []

    if verbose:
        bui    = bui_to_class(Path(csv_path).name)
        bname  = CLASS_NAMES.get(bui, "Unknown class")
        print(f"\n{'─'*80}")
        print(f"REAL-TIME SCAN: {Path(csv_path).name}  [{bname}]")
        print(f"{'─'*80}")
        print(f"{'Seg':>4}  {'Label':<22}  "
              f"{'CalibConf':>9}  {'TScore':>7}  "
              f"{'Epistemic':>9}  {'Aleatoric':>9}  "
              f"{'Margin':>7}  {'CredN':>5}")
        print("─" * 80)

    for seg_idx, start in enumerate(range(0, len(amp) - WINDOW_SIZE, STEP_SIZE)):
        if seg_idx >= max_segments:
            break
        segment = amp[start: start + WINDOW_SIZE]
        fv      = safe_extract(segment)
        res     = classify_signal(fv, return_bayes=True)
        decisions.append(res)

        if verbose:
            b    = res["bayesian"]
            icon = ICONS.get(res["label"], "❓")
            print(
                f"{seg_idx+1:>4}  {icon} {res['label']:<20}  "
                f"{b.get('calibrated_confidence',0):>9.4f}  "
                f"{b.get('threat_score',0):>7.4f}  "
                f"{b.get('epistemic_uncertainty',0):>9.4f}  "
                f"{b.get('aleatoric_uncertainty',0):>9.4f}  "
                f"{b.get('margin',0):>7.4f}  "
                f"{b.get('n_credible_classes',0):>5}"
            )

    if verbose and decisions:
        lc    = Counter(d["label"] for d in decisions)
        parts = "  ".join(f"{ICONS.get(k,'?')} {k}: {v}"
                          for k, v in sorted(lc.items()))
        print("─" * 80)
        print(f"SUMMARY ({len(decisions)} segs): {parts}")
        n_thr = sum(1 for d in decisions if "THREAT" in d.get("label", ""))
        if n_thr:
            print(f"⚠️  ALERT — {n_thr} threat segment(s) detected!")

    return decisions


# Run on first 3 real files
print(f"\n{'='*65}")
print("REAL-TIME DETECTION DEMO (first 3 files in dataset)")
print(f"{'='*65}")
try:
    _demo_files = []
    for _files in discover_files(DATA_DIR).values():
        _demo_files.extend(_files[:1])
    for _fp in _demo_files[:3]:
        realtime_detection_loop(str(_fp), max_segments=8)
except Exception as _e:
    print(f"  (Demo skipped — data dir not accessible: {_e})")


# ════════════════════════════════════════════════════════════════════════════
# SECTION 18 — FULL SYSTEM EVALUATION ON HELD-OUT TEST SET
# ════════════════════════════════════════════════════════════════════════════

print(f"\n{'='*65}")
print("FULL SYSTEM EVALUATION — held-out real test set")
print(f"{'='*65}")

X_te_raw = scaler.inverse_transform(X_te_s)   # recover raw scale for emitter_hash + classify

test_decs = []
for i in range(len(X_te_raw)):
    dec = classify_signal(X_te_raw[i], return_bayes=True)
    dec["true_class"] = CLASSES_PRESENT[y_te[i]]
    test_decs.append(dec)

test_df = pd.DataFrame(test_decs)

# Extract Bayesian fields into flat columns for analysis
for col in ["calibrated_confidence", "predictive_entropy",
            "epistemic_uncertainty", "aleatoric_uncertainty",
            "margin", "threat_score", "n_credible_classes",
            "max_confidence", "is_novel", "winner"]:
    test_df[col] = test_df["bayesian"].apply(
        lambda b: b.get(col, None) if isinstance(b, dict) else None
    )

known_mask  = ~test_df["label"].isin(
    ["POTENTIAL_THREAT", "CONFIRMED_THREAT", "UNKNOWN_MONITOR", "SAFE_NEW_DRONE"]
)
correct     = (test_df.loc[known_mask, "winner"] ==
               test_df.loc[known_mask, "true_class"]).mean() if known_mask.sum() else 0
false_alarm = test_df["label"].isin(
    ["POTENTIAL_THREAT", "CONFIRMED_THREAT"]
).mean()
bg_recall   = (
    test_df[test_df["true_class"] == CLASS_NAMES[0]]["label"]
    .eq("BACKGROUND").mean()
    if (test_df["true_class"] == CLASS_NAMES[0]).any() else 0.0
)

# Confidence statistics
high_conf = test_df["calibrated_confidence"] >= 0.7
print(f"\n  Test set size                      : {len(test_df):,}")
print(f"  Known-drone classification acc     : {correct:.2%}  "
      f"({known_mask.sum()} non-alarmed samples)")
print(f"  High-confidence predictions (≥0.70): {high_conf.sum()} "
      f"({high_conf.mean():.1%} of test set)")
print(f"  False alarm rate (known→threat)    : {false_alarm:.2%}")
print(f"  Background recall                  : {bg_recall:.2%}")

print(f"\n  Bayesian uncertainty statistics:")
for col, label in [
    ("calibrated_confidence",  "Mean calibrated confidence"),
    ("predictive_entropy",     "Mean predictive entropy"),
    ("epistemic_uncertainty",  "Mean epistemic uncertainty"),
    ("aleatoric_uncertainty",  "Mean aleatoric uncertainty"),
    ("margin",                 "Mean decision margin"),
]:
    vals = test_df[col].dropna()
    print(f"    {label:<35} {vals.mean():.4f}  (std={vals.std():.4f})")

print(f"\n  Label distribution on test set:")
for lbl, cnt in test_df["label"].value_counts().items():
    pct = cnt / len(test_df)
    bar = "█" * int(pct * 40)
    print(f"    {ICONS.get(lbl,'?')} {lbl:<25} {cnt:>5}  ({pct:.1%})  {bar}")

test_df.to_csv("system_test_decisions.csv", index=False)
print("\n  ✓ Audit trail → system_test_decisions.csv")


# ════════════════════════════════════════════════════════════════════════════
# SECTION 19 — VISUALISATIONS  (8-panel dashboard)
# ════════════════════════════════════════════════════════════════════════════

C = {
    "friendly":    "#1D9E75",
    "threat":      "#D85A30",
    "background":  "#888780",
    "safe_new":    "#3B82F6",
    "monitor":     "#F59E0B",
    "trusted_new": "#6366F1",
    "bayesian":    "#7F77DD",
    "rf":          "#1D9E75",
}

fig = plt.figure(figsize=(26, 24))
gs  = gridspec.GridSpec(3, 3, figure=fig, hspace=0.58, wspace=0.42)

# ── Panel 1: Feature importances (top 20) ────────────────────────────────
ax1   = fig.add_subplot(gs[0, :2])
top_n = min(20, N_FEATURES)
ti    = np.argsort(importances)[::-1][:top_n]
ti_nm = [FEATURE_NAMES[i] for i in ti]
pcols = ["#7F77DD" if "energy" in n else
          "#3B82F6" if any(k in n for k in
          ("freq","bandwidth","entropy","centroid","spread","rolloff","psd")) else
          "#1D9E75" for n in ti_nm]
ax1.barh(ti_nm[::-1], importances[ti][::-1], color=pcols[::-1], height=0.72)
ax1.set_xlabel("Feature importance (Gini)", fontsize=11)
ax1.set_title("Top Feature Importances — RF Classifier (Frozen)",
              fontsize=12, fontweight="500")
ax1.legend(handles=[
    Patch(facecolor="#1D9E75", label="Amplitude / Hilbert"),
    Patch(facecolor="#3B82F6", label="Spectral / PSD"),
    Patch(facecolor="#7F77DD", label="Band energy"),
], fontsize=9, loc="lower right")
ax1.tick_params(labelsize=9)

# ── Panel 2: Bayesian confidence distribution ─────────────────────────────
ax2 = fig.add_subplot(gs[0, 2])
for lbl, col, alpha in [
    ("FRIENDLY_DRONE", C["friendly"], 0.8),
    ("BACKGROUND",     C["background"], 0.6),
]:
    mask_l = test_df["label"] == lbl
    vals   = test_df.loc[mask_l, "calibrated_confidence"].dropna()
    if len(vals):
        ax2.hist(vals, bins=25, alpha=alpha, density=True, color=col, label=lbl)
ax2.axvline(CONFIDENCE_THRESHOLD, color="black", ls="--", lw=2,
            label=f"Gate={CONFIDENCE_THRESHOLD}")
ax2.set_xlabel("Calibrated confidence", fontsize=10)
ax2.set_ylabel("Density", fontsize=10)
ax2.set_title("Calibrated Confidence\n(Known Drone Predictions)", fontsize=11, fontweight="500")
ax2.legend(fontsize=8)

# ── Panel 3: Epistemic vs Aleatoric uncertainty scatter ───────────────────
ax3 = fig.add_subplot(gs[1, 0])
label_color_map = {
    "FRIENDLY_DRONE":    C["friendly"],
    "BACKGROUND":        C["background"],
    "POTENTIAL_THREAT":  C["threat"],
    "CONFIRMED_THREAT":  C["threat"],
    "SAFE_NEW_DRONE":    C["safe_new"],
    "UNKNOWN_MONITOR":   C["monitor"],
    "TRUSTED_NEW_DRONE": C["trusted_new"],
}
for lbl, col in label_color_map.items():
    m = test_df["label"] == lbl
    if m.sum() == 0:
        continue
    ax3.scatter(
        test_df.loc[m, "aleatoric_uncertainty"],
        test_df.loc[m, "epistemic_uncertainty"],
        c=col, alpha=0.4, s=12, label=f"{lbl} (n={m.sum()})"
    )
ax3.set_xlabel("Aleatoric uncertainty (irreducible)", fontsize=10)
ax3.set_ylabel("Epistemic uncertainty (model gap)", fontsize=10)
ax3.set_title("Uncertainty Decomposition\n(Test Set)", fontsize=11, fontweight="500")
ax3.legend(fontsize=7, ncol=1)

# ── Panel 4: Trust accumulation — safe emitter ───────────────────────────
ax4 = fig.add_subplot(gs[1, 1])
safe_key = "Harmless_Surveyor"
if safe_key in sim_results:
    trust_curve = [d["trust_score"] for d in sim_results[safe_key]]
    calib_curve = [d["bayesian"].get("calibrated_confidence", 0) for d in sim_results[safe_key]]
    ax4.plot(range(1, len(trust_curve)+1), trust_curve,
             color=C["safe_new"], lw=2.5, marker="o", ms=6, label="Trust score")
    ax4.plot(range(1, len(calib_curve)+1), calib_curve,
             color=C["bayesian"], lw=2, marker="s", ms=5, ls="--", label="Calibrated conf")
    ax4.axhline(0.5, color="gray", ls=":", lw=1, label="Promotion ~0.5")
    ax4.set_ylim(0, 1.05)
    ax4.set_xlabel("Observation #", fontsize=10)
    ax4.set_ylabel("Score", fontsize=10)
    ax4.set_title(f"Trust Accumulation\n({safe_key})", fontsize=11, fontweight="500")
    ax4.legend(fontsize=8)

# ── Panel 5: Threat evolution — threat emitter ────────────────────────────
ax5 = fig.add_subplot(gs[1, 2])
thr_key = "DJI_Neo_Threat"
if thr_key in sim_results:
    ts_curve  = [d["bayesian"].get("threat_score",  0) for d in sim_results[thr_key]]
    ep_curve  = [d["bayesian"].get("epistemic_uncertainty", 0) for d in sim_results[thr_key]]
    ax5.plot(range(1, len(ts_curve)+1), ts_curve,
             color=C["threat"], lw=2.5, marker="s", ms=6, label="Threat score")
    ax5.plot(range(1, len(ep_curve)+1), ep_curve,
             color="#888780", lw=2, marker="^", ms=5, ls="--", label="Epistemic uncertainty")
    ax5.axhline(THREAT_THRESHOLD, color="black", ls="--", lw=1.5,
                label=f"Threat threshold={THREAT_THRESHOLD:.3f}")
    ax5.set_ylim(0, 1.05)
    ax5.set_xlabel("Observation #", fontsize=10)
    ax5.set_ylabel("Score", fontsize=10)
    ax5.set_title(f"Threat Detection Over Time\n({thr_key})", fontsize=11, fontweight="500")
    ax5.legend(fontsize=8)

# ── Panel 6: Online GNB learning curve ───────────────────────────────────
ax6 = fig.add_subplot(gs[2, 0])
if f1_curve:
    ax6.plot(range(1, len(f1_curve)+1), f1_curve,
             color=C["bayesian"], lw=2.5, marker="o", ms=4)
    ax6.axhline(max(f1_curve), color="gray", ls="--", lw=1,
                label=f"Max F1={max(f1_curve):.4f}")
    ax6.set_ylim(0, 1.05)
    ax6.set_xlabel("Batch #", fontsize=10)
    ax6.set_ylabel("F1-macro", fontsize=10)
    ax6.set_title("Online GNB — Bayesian Learning Curve\n(partial_fit, no full retrain)",
                  fontsize=11, fontweight="500")
    ax6.legend(fontsize=8)

# ── Panel 7: Confusion matrix ─────────────────────────────────────────────
ax7     = fig.add_subplot(gs[2, 1])
known_s = test_df[known_mask & test_df["winner"].notna()].copy()
if len(known_s) > 0:
    pres = sorted(set(known_s["true_class"]) | set(known_s["winner"]))
    cm   = confusion_matrix(known_s["true_class"], known_s["winner"], labels=pres)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=pres, yticklabels=pres,
                ax=ax7, cbar=False, annot_kws={"size": 9})
    ax7.set_xlabel("Predicted", fontsize=10)
    ax7.set_ylabel("True", fontsize=10)
    ax7.set_title("Confusion Matrix\n(RF+GNB Combined Classifier)",
                  fontsize=11, fontweight="500")
    ax7.tick_params(labelsize=8)

# ── Panel 8: Model benchmark comparison ───────────────────────────────────
ax8 = fig.add_subplot(gs[2, 2])
bm_names = [k for k, v in bench_results.items() if v.get("f1_macro") is not None]
bm_f1    = [bench_results[m]["f1_macro"] for m in bm_names]
# Add RF and GNB to benchmark for complete picture
all_names = ["RandomForest", "GaussianNB (Bayesian)"] + bm_names
all_f1    = [f1_rf, f1_gnb] + bm_f1
bm_cols   = [C["rf"], C["bayesian"]] + [
    C["bayesian"] if "NB" in n or "GMM" in n or "Dirichlet" in n else C["rf"]
    for n in bm_names
]
ax8.barh(all_names, all_f1, color=bm_cols, height=0.65)
for i, v in enumerate(all_f1):
    ax8.text(v + 0.003, i, f"{v:.3f}", va="center", fontsize=9)
ax8.set_xlim(0, max(all_f1) * 1.18 if all_f1 else 1)
ax8.set_xlabel("F1-macro", fontsize=10)
ax8.set_title("Model Benchmark\n(green=discriminative, purple=Bayesian)",
              fontsize=11, fontweight="500")
ax8.tick_params(labelsize=8)
ax8.legend(handles=[
    Patch(facecolor=C["rf"],       label="Discriminative"),
    Patch(facecolor=C["bayesian"], label="Bayesian / Generative"),
], fontsize=8)

fig.suptitle(
    "Continual RF Drone Monitoring System — Production Grade\n"
    "Goal: Confidence-first prediction  |  Dual Bayesian Engine (RF + GNB)  |  "
    "No retraining after deployment",
    fontsize=13, fontweight="600",
)
plt.savefig("continual_monitor_dashboard.png", dpi=150, bbox_inches="tight")
plt.close()
print("\n✓ 8-panel dashboard saved → continual_monitor_dashboard.png")


# ════════════════════════════════════════════════════════════════════════════
# SECTION 20 — SAVE STATE + FINAL SUMMARY
# ════════════════════════════════════════════════════════════════════════════

fp_db.save()
pd.DataFrame({"batch": range(1, len(f1_curve)+1),
               "f1_macro": f1_curve}).to_csv("online_gnb_learning_curve.csv", index=False)

print(f"\n{'='*72}")
print("CONTINUAL RF DRONE MONITORING SYSTEM — FINAL SUMMARY")
print(f"{'='*72}")
print(f"""
DATASET
  Source     : data.mendeley.com/datasets/f4c2b4n755/1
  Segments   : {len(df_real):,} (balanced, real data only, no augmentation)
  Window     : {WINDOW_SIZE} samples  |  Step: {STEP_SIZE}
  Features   : {N_FEATURES} (amplitude + Hilbert + spectral + inst.freq + band energy)
  Classes    : {N_CLS}  ({" | ".join(CLASSES_PRESENT)})
  SMOTE      : applied to training split only (k={k_smote})

STAGE 1 — FEATURE EXTRACTION (30 features)
  Group A (8) : raw amplitude statistics        — power, shape, dynamics
  Group B (6) : Hilbert analytic envelope       — IQ_corr, power_dB, env stats
  Group C (8) : Welch PSD spectral              — carrier, BW, entropy, centroid
  Group D (4) : instantaneous frequency         — FM modulation rate & kurtosis
  Group E (4) : band energy ratios              — scale-invariant spectral shape

STAGE 2 — DUAL BAYESIAN CLASSIFIER  (FROZEN after training)
  Random Forest        OOB={rf_clf.oob_score_:.4f}  acc={acc_rf:.4f}  F1={f1_rf:.4f}
  GaussianNaiveBayes   acc={acc_gnb:.4f}  F1={f1_gnb:.4f}  [TRUE Bayesian posterior]
  Combined posterior   = geometric mean √(P_RF × P_GNB)
  Confidence gate      : calibrated_conf ≥ {CONFIDENCE_THRESHOLD} → known
  Online GNB           : partial_fit() Bayesian update, {len(f1_curve)} batches

BAYESIAN CONFIDENCE BREAKDOWN (every prediction):
  max_posterior        P(winning class | x)          from combined model
  predictive_entropy   H[y|x] / log(N)  ∈ [0,1]    total uncertainty
  epistemic_uncertainty model knowledge gap ∈ [0,1]  → should we trust this?
  aleatoric_uncertainty irreducible signal noise ∈ [0,1] → fundamental ambiguity?
  margin               P(winner) − P(second_best)   → decisiveness
  calibrated_confidence P(winner) × (1+margin) / 2  → adjusted final confidence
  n_credible_classes   classes summing to 95% posterior mass

STAGE 3 — ANOMALY DETECTION  (4 detectors, FROZEN)
  Mahalanobis  (w={ANOMALY_WEIGHTS['mahal']}) class-conditional covariance distance
  Per-Class GMM(w={ANOMALY_WEIGHTS['gmm']}) per-class density model
  IsoForest    (w={ANOMALY_WEIGHTS['isoforest']}) geometric isolation (no distribution assumption)
  GNB-LL       (w={ANOMALY_WEIGHTS['gnb']}) Bayesian marginal log-likelihood

STAGE 4 — THREAT SCORING
  Ensemble threshold   : {THREAT_THRESHOLD:.4f}  (97th-percentile of training)
  Dirichlet Process GMM: active_clusters={active_clusters} (non-parametric Bayesian)

STAGE 5 — TEMPORAL TRUST  (harmonic mean of 3 criteria)
  obs_trust   = sigmoid(seen / {TRUST_MIN_OBSERVATIONS})
  stab_trust  = 1 − variance / {TRUST_MAX_VARIANCE}
  safe_trust  = 1 − mean_threat_score
  {temporal_tracker.summary()}

STAGE 6 — RF FINGERPRINT DATABASE
  {fp_db.summary()}
  Matching     : cosine similarity (scale-invariant, threshold={SIMILARITY_THRESHOLD})
  Persistence  : {DB_PATH}

STAGE 7 — DECISION LABELS
  🟢 FRIENDLY_DRONE    calibrated_conf ≥ {CONFIDENCE_THRESHOLD}, known class
  ⚪ BACKGROUND        classifier → Background class
  🔷 TRUSTED_NEW_DRONE cosine sim ≥ {SIMILARITY_THRESHOLD} to trusted fingerprint
  🔴 POTENTIAL_THREAT  threat score ≥ threshold, first/second sighting
  🚨 CONFIRMED_THREAT  threat score ≥ threshold, seen ≥ 3 times
  🔵 SAFE_NEW_DRONE    stable + frequent + low-threat → promoted to trusted DB
  🟡 UNKNOWN_MONITOR   insufficient evidence yet

EVALUATION (held-out real test set)
  Known-drone accuracy   : {correct:.2%}  ({known_mask.sum()} non-alarmed samples)
  False alarm rate       : {false_alarm:.2%}
  Background recall      : {bg_recall:.2%}

NO-RETRAINING GUARANTEE
  RandomForest weights   : frozen after Section 9
  GNB model params       : frozen after Section 9
  Anomaly detector params: frozen after Section 11
  StandardScaler params  : frozen after Section 8
  New drones handled by  : temporal tracker + cosine fingerprint DB ONLY
  DB growth              : stores mean feature vectors, zero weight updates

OUTPUT FILES
  {OUTPUT_CSV:<50} — balanced feature dataset
  benchmark_results.csv                            — 7-model comparison
  system_test_decisions.csv                        — full audit trail
  online_gnb_learning_curve.csv                    — Bayesian update history
  {DB_PATH:<50} — fingerprint database
  continual_monitor_dashboard.png                  — 8-panel diagnostic chart
""")
print("=" * 72)
print("System ready for deployment.")


✓ All imports ready
✓ Feature pipeline ready — 30 features per segment

STAGE 1 — BALANCED DATASET EXTRACTION
  window=200  step=100  target=4,500  fs=10 MHz

Discovering files…
  Layout A detected (named subfolders):
    [0] Background RF activities              82 files
    [1] AR drone                             162 files
    [2] Phantom drone                         42 files

  3 classes  |  1,500 segments per class

  [0] Background RF activities  (quota=1,500):
    00000L_7.csv                                   +   98  total=    98/1500
    00000H_10.csv                                  +   98  total=   196/1500
    00000H_12.csv                                  +   98  total=   294/1500
    00000L_19.csv                                  +   98  total=   392/1500
    00000H_27.csv                                  +   98  total=   490/1500
    00000L_22.csv                                  +   98  total=   588/1500
    00000H_33.csv                                  +   98  total=